In [3]:
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import pandas as pd

# Define the scope
scope = [
    'https://spreadsheets.google.com/feeds',
    'https://www.googleapis.com/auth/drive'
]

# Authenticate using the credentials file
creds = ServiceAccountCredentials.from_json_keyfile_name('credentials.json', scope)
client = gspread.authorize(creds)

# Open the Google Sheet by title
try:
    # Get the the names of all sheets in the spreadsheet
    spreadsheet = client.open("Aesthetico")
    print("Available sheets:", spreadsheet.worksheets())
    sheet = client.open("Aesthetico").sheet1  # Use sheet1 or specify name
    print("Spreadsheet opened successfully!")
except gspread.exceptions.SpreadsheetNotFound:
    print("Spreadsheet not found. Check the name and sharing settings.")

# # Read data
# data = sheet.get_all_records()  # Returns a list of dictionaries
# df = pd.DataFrame(data)  # Convert to pandas DataFrame
# print(df.head())   

Available sheets: [<Worksheet 'DASHBOARD' id:775962956>, <Worksheet 'Inventory' id:1954917366>, <Worksheet 'Summary Inventory' id:1666316797>, <Worksheet 'Input Inventory' id:323391169>, <Worksheet 'Sales' id:1755340049>, <Worksheet 'Sales Raw' id:1009241507>, <Worksheet 'Expenses' id:774591608>, <Worksheet 'Product Purchase' id:1346236829>, <Worksheet 'DashBoard Calculation' id:2115879885>, <Worksheet 'Daily Inventory' id:1635343157>, <Worksheet 'Reference Inventory' id:793239626>, <Worksheet 'Information' id:413721828>]
Spreadsheet opened successfully!


In [4]:
import gspread
from google.oauth2.service_account import Credentials
import pandas as pd

# Define scopes
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]

# Authenticate
creds = Credentials.from_service_account_file("credentials.json", scopes=SCOPES)
client = gspread.authorize(creds)

# Step 1: Verify connection by listing all accessible spreadsheets
print("=== Connection Test ===")
spreadsheets = client.openall()
print(f"Credentials are working. Found {len(spreadsheets)} accessible spreadsheet(s):")
for s in spreadsheets:
    print(f"  - {s.title} (ID: {s.id})")

# Step 2: Open the spreadsheet and list all sheets (tabs)
print("\n=== Opening 'Aesthetico' Spreadsheet ===")
try:
    spreadsheet = client.open("Aesthetico")
    print(f"Spreadsheet opened: {spreadsheet.title}")
    print(f"Spreadsheet ID: {spreadsheet.id}")
    print(f"Number of sheets: {len(spreadsheet.worksheets())}")
    for ws in spreadsheet.worksheets():
        print(f"  - Sheet: '{ws.title}' (Rows: {ws.row_count}, Cols: {ws.col_count})")
except gspread.exceptions.SpreadsheetNotFound:
    print("ERROR: Spreadsheet 'Aesthetico' not found.")
    print("Make sure you shared the spreadsheet with:")
    print("  sheet-accessor@aesthetico-inventory.iam.gserviceaccount.com")
    exit(1)

# Step 3: Read data from the 'Sales' sheet
print("\n=== Reading 'Sales' Sheet ===")
sheet = spreadsheet.worksheet("Sales")
data = sheet.get_all_records()

if data:
    df = pd.DataFrame(data)
    print(f"Read {len(df)} rows and {len(df.columns)} columns.")
    print(f"Columns: {list(df.columns)}")
    print("\nFirst 5 rows:")
    print(df.head().to_string())
else:
    print("Sheet is empty or has no data rows.")


=== Connection Test ===
Credentials are working. Found 1 accessible spreadsheet(s):
  - Aesthetico (ID: 1Y5khuotkswYdPA_g5NZJuk9BUrhmR6jtmVPkdsYszS8)

=== Opening 'Aesthetico' Spreadsheet ===
Spreadsheet opened: Aesthetico
Spreadsheet ID: 1Y5khuotkswYdPA_g5NZJuk9BUrhmR6jtmVPkdsYszS8
Number of sheets: 12
  - Sheet: 'DASHBOARD' (Rows: 50503, Cols: 26)
  - Sheet: 'Inventory' (Rows: 1004, Cols: 25)
  - Sheet: 'Summary Inventory' (Rows: 999, Cols: 26)
  - Sheet: 'Input Inventory' (Rows: 908, Cols: 26)
  - Sheet: 'Sales' (Rows: 2738, Cols: 27)
  - Sheet: 'Sales Raw' (Rows: 2316, Cols: 25)
  - Sheet: 'Expenses' (Rows: 996, Cols: 24)
  - Sheet: 'Product Purchase' (Rows: 1000, Cols: 29)
  - Sheet: 'DashBoard Calculation' (Rows: 1008, Cols: 26)
  - Sheet: 'Daily Inventory' (Rows: 1000, Cols: 26)
  - Sheet: 'Reference Inventory' (Rows: 1000, Cols: 29)
  - Sheet: 'Information' (Rows: 1001, Cols: 26)

=== Reading 'Sales' Sheet ===
Read 175 rows and 18 columns.
Columns: ['Order ID', 'Date', 'Custome

In [5]:
df.columns

Index(['Order ID', 'Date', 'Customer Name', 'Product ID', 'Quantity',
       'Total Price ', 'Discount', 'Discounted Price', 'Delivery Platform',
       'Delivery Cost', 'Payable Amount', 'Advance Payment', 'Due Payment',
       'Order Status', 'Sales By', 'Return Reason', 'Dispatch Date',
       'Consignment ID'],
      dtype='object')

In [25]:
# get those rows which Consignment ID has value
df_secnd=df[df['Consignment ID']!='']

In [26]:
df_secnd

,Order ID,Date,Customer Name,Product ID,Quantity,Total Price,Discount,Discounted Price,Delivery Platform,Delivery Cost,Payable Amount,Advance Payment,Due Payment,Order Status,Sales By,Return Reason,Dispatch Date,Consignment ID
171,ORD#20260201-01478,2/1/2026,MD. Nayim Hossen,PRD-023,1,380,20,360,Pathao Inside,70,430,,430,,Durba,,,DA010226MAN7CG
174,ORD#20260201-01481,2/1/2026,শুভজিত মন্ডল,PRD-008,1,350,,350,Pathao Outside,110,460,,460,,Durba,,,DA010226BYHR3Y


In [11]:
from pathao_courier import PathaoCourier

pathao = PathaoCourier()

In [22]:
def get_status(consignment_id):
    order_info = pathao.get_order_info(consignment_id)
    return order_info['order_status']

In [27]:
df_secnd['Order Status'] = df_secnd['Consignment ID'].apply(get_status)

C:\Users\fuads\AppData\Local\Temp\ipykernel_20532\1515330549.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_secnd['Order Status'] = df_secnd['Consignment ID'].apply(get_status)


In [28]:
df_secnd

,Order ID,Date,Customer Name,Product ID,Quantity,Total Price,Discount,Discounted Price,Delivery Platform,Delivery Cost,Payable Amount,Advance Payment,Due Payment,Order Status,Sales By,Return Reason,Dispatch Date,Consignment ID
171,ORD#20260201-01478,2/1/2026,MD. Nayim Hossen,PRD-023,1,380,20,360,Pathao Inside,70,430,,430,Delivered,Durba,,,DA010226MAN7CG
174,ORD#20260201-01481,2/1/2026,শুভজিত মন্ডল,PRD-008,1,350,,350,Pathao Outside,110,460,,460,Delivered,Durba,,,DA010226BYHR3Y


In [32]:
from steadfast_courier import SteadfastCourier

sf = SteadfastCourier()

# By consignment ID (numeric)
order = sf.get_order_by_consignment("216293979")
print(order["delivery_status"])  # e.g. "delivered", "pending", "cancelled"

# # By invoice
# order = sf.get_order_by_invoice("INV-001")

# # By tracking code
# order = sf.get_order_by_tracking("TRACK-001")

pending


In [33]:
order

{'status': 200, 'delivery_status': 'pending'}